Copyright (c) Microsoft Corporation. All rights reserved.  
Licensed under the MIT License.

# Bert Quantization with ONNX Runtime on CPU

In this tutorial, we will load a fine tuned [HuggingFace BERT](https://huggingface.co/transformers/) model trained with [PyTorch](https://pytorch.org/) for [Microsoft Research Paraphrase Corpus (MRPC)](https://www.microsoft.com/en-us/download/details.aspx?id=52398) task , convert the model to ONNX, and then quantize PyTorch and ONNX model respectively. Finally, we will demonstrate the performance, accuracy and model size of the quantized PyTorch and OnnxRuntime model in the [General Language Understanding Evaluation benchmark (GLUE)](https://gluebenchmark.com/)

## 0. Prerequisites ##

If you have Jupyter Notebook, you can run this notebook directly with it. You may need to install or upgrade [PyTorch](https://pytorch.org/), [OnnxRuntime](https://microsoft.github.io/onnxruntime/), [transformers](https://huggingface.co/transformers/) and other required packages.

Otherwise, you can setup a new environment. First, install [AnaConda](https://www.anaconda.com/distribution/). Then open an AnaConda prompt window and run the following commands:

```console
conda create -n cpu_env python=3.13
conda activate cpu_env
conda install jupyter
jupyter notebook
```
The last command will launch Jupyter Notebook and we can open this notebook in browser to continue.

### 0.1 Install packages
Let's install nessasary packages to start the tutorial. We will install PyTorch 2.9, OnnxRuntime 1.23, latest ONNX, transformers, and scikit-learn, datasets, latest evaluate.

In [4]:
# Install or upgrade PyTorch 2.9.0 and OnnxRuntime 1.23.0 for CPU-only.
import sys
!{sys.executable} -m pip install torch==2.9.0 torchvision==0.24.0 torchaudio==2.9.0 --index-url https://download.pytorch.org/whl/cpu
!{sys.executable} -m pip install --upgrade onnxruntime==1.23.0

# Install other packages used in this notebook.
!{sys.executable} -m pip install --upgrade transformers onnx datasets evaluate scipy scikit-learn
!{sys.executable} -m pip install accelerate>=0.26.0

Looking in indexes: https://download.pytorch.org/whl/cpu
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ---------------------------------------- 0.0/38.5 MB ? eta -:--:--
     -- ------------------------------------- 2.4/38.5 MB 11.9 MB/s eta 0:00:04
     ----- ---------------------------------- 5.0/38.5 MB 11.9 MB/s eta 0:00:03
     ------- -------------------------------- 7.3/38.5 MB 11.9 MB/s eta 0:00:03
     ---------- ----------------------------- 9.7/38.5 MB 11.8 MB/s eta 0:00:03
     ------------ -------------------------- 12.3/38.5 MB 11.8 MB/s eta 0:00:03
     -------------- ------------------------ 14.7/38.5 MB 11.8 MB/s eta 0:00:03
     ----------------- --------------------- 17.0/38.5 MB 11.8 MB/s eta 0:00:02
     ------------------- ------------------- 19.7/38.5 MB 11.8 MB/s eta 0:00:02
     ---------------------- ---------------- 22.0/38.5 MB 11.8 MB/s eta 0:00:02
     ------------------------

### 0.2 Download GLUE data and Fine-tune BERT model for MPRC task
HuggingFace [text-classification examples]( https://github.com/huggingface/transformers/tree/master/examples/text-classification) shows details on how to fine-tune a MPRC tack with GLUE data.

#### Firstly, Let's download the GLUE data with download_glue_data.py [script](https://github.com/huggingface/transformers/blob/master/utils/download_glue_data.py) and unpack it to directory glue_data under current directory.

In [10]:
!wget https://raw.githubusercontent.com/huggingface/transformers/main/utils/download_glue_data.py
!python download_glue_data.py --data_dir="glue_data" --tasks="MRPC"

Processing MRPC...
Local MRPC data not specified, downloading data from https://dl.fbaipublicfiles.com/senteval/senteval_data/msr_paraphrase_train.txt
	Completed!


#### Next, we can fine-tune the model based on the [MRPC example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/README.md) with command like:

`
export GLUE_DIR=./glue_data
export TASK_NAME=MRPC
export OUT_DIR=./$TASK_NAME/
python ./run_glue.py \
    --model_type bert \
    --model_name_or_path bert-base-uncased \
    --task_name $TASK_NAME \
    --do_train \
    --do_eval \
    --do_lower_case \
    --data_dir $GLUE_DIR/$TASK_NAME \
    --max_seq_length 128 \
    --per_gpu_eval_batch_size=8   \
    --per_gpu_train_batch_size=8   \
    --learning_rate 2e-5 \
    --num_train_epochs 3.0 \
    --save_steps 100000 \
    --output_dir $OUT_DIR
`

In order to save time, we download the fine-tuned BERT model for MRPC task by PyTorch from:https://download.pytorch.org/tutorial/MRPC.zip.

In [14]:
!curl https://download.pytorch.org/tutorial/MRPC.zip --output MPRC.zip
!unzip -n MPRC.zip

Archive:  MPRC.zip
   creating: MRPC/                 
 extracting: MRPC/added_tokens.json  
  inflating: MRPC/tokenizer_config.json  
  inflating: MRPC/special_tokens_map.json  
  inflating: MRPC/config.json        
  inflating: MRPC/training_args.bin  
  inflating: MRPC/vocab.txt          
  inflating: MRPC/pytorch_model.bin  


## 1.Load and quantize model with PyTorch

In this section, we will load the fine-tuned model with PyTorch, quantize it and measure the performance. 

### 1.1 Import modules and set global configurations

In this step, we import the necessary PyTorch, transformers and other necessary modules for the tutorial, and then set up the global configurations, like data & model folder, GLUE task settings, thread settings, warning settings and etc.

In [9]:
import logging
import os
import sys
import torch
from collections import Counter
from argparse import Namespace

import evaluate
import numpy as np
from datasets import load_dataset

from transformers import (
    EvalPrediction,
    PretrainedConfig,
    Trainer,
    TrainingArguments,
    default_data_collator,
    set_seed,
)
from transformers import (BertForSequenceClassification, BertTokenizer,)
from transformers.utils import check_min_version


# Will error if the minimal version of Transformers is not installed. Remove at your own risks.
check_min_version("4.57.0")

task_to_keys = {
    "cola": ("sentence", None),
    "mnli": ("premise", "hypothesis"),
    "mrpc": ("sentence1", "sentence2"),
    "qnli": ("question", "sentence"),
    "qqp": ("question1", "question2"),
    "rte": ("sentence1", "sentence2"),
    "sst2": ("sentence", None),
    "stsb": ("sentence1", "sentence2"),
    "wnli": ("sentence1", "sentence2"),
}

logger = logging.getLogger(__name__)

configs = Namespace()

# The output directory for the fine-tuned model, $OUT_DIR.
configs.output_dir = "./MRPC/"

# The data directory for the MRPC task in the GLUE benchmark, $GLUE_DIR/$TASK_NAME.
configs.data_dir = "./glue_data/MRPC"

# The model name or path for the pre-trained model.
configs.model_name_or_path = "bert-base-uncased"
# The maximum length of an input sequence
configs.max_seq_length = 128

# Prepare GLUE task.
configs.task_name = "MRPC".lower()
configs.device = "cpu"

# Where do you want to store the pretrained models downloaded from huggingface.co
configs.cache_dir = os.path.join(configs.data_dir, 'cached_eval_{}_{}_{}'.format(
    list(filter(None, configs.model_name_or_path.split('/'))).pop(),
    str(configs.max_seq_length),
    str(configs.task_name)))

training_args = TrainingArguments(
    output_dir=configs.output_dir,
    do_eval=True, use_cpu=True)

# Setup logging
logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    datefmt="%m/%d/%Y %H:%M:%S",
    handlers=[logging.StreamHandler(sys.stdout)],
)

# Set seed before initializing model.
set_seed(training_args.seed)

### 1.2 Load and quantize the fine-tuned BERT model with PyTorch 
In this step, we load the fine-tuned BERT model, and quantize it with PyTorch's dynamic quantization. And show the model size comparison between full precision and quantized model.

In [10]:
def print_size_of_model(model):
    torch.save(model.state_dict(), "temp.p")
    print('Size (MB):', os.path.getsize("temp.p")/(1024*1024))
    os.remove('temp.p')

# Load pretrained model and tokenizer
tokenizer = BertTokenizer.from_pretrained(configs.output_dir)

model = BertForSequenceClassification.from_pretrained(configs.output_dir)
model.to(configs.device)

quantized_model = torch.ao.quantization.quantize_dynamic(
    model, {torch.nn.Linear}, dtype=torch.qint8
)

print_size_of_model(model)

print_size_of_model(quantized_model)

C:\Users\18220\AppData\Local\Temp\ipykernel_14996\3014477203.py:12: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.ao.quantization.quantize_dynamic(


Size (MB): 417.710205078125
Size (MB): 173.0729637145996


### 1.3 Evaluate the accuracy and performance of PyTorch quantization
This section reused the tokenize and evaluation function from [Huggingface](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification/run_glue.py).

In [12]:
def evaluate_model(model, tokenizer):
    # Preprocessing the raw_datasets
    raw_datasets = load_dataset("glue", configs.task_name)

    # Labels
    is_regression = configs.task_name == "stsb"
    if not is_regression:
        label_list = raw_datasets["train"].features["label"].names
        num_labels = len(label_list)
    else:
        num_labels = 1
        
    sentence1_key, sentence2_key = task_to_keys[configs.task_name]

    # Padding strategy
    padding = "max_length"

    # Some models have set the order of the labels to use, so let's make sure we do use it.
    label_to_id = None
    if (
        model.config.label2id != PretrainedConfig(num_labels=num_labels).label2id
        and not is_regression
    ):
        # Some have all caps in their config, some don't.
        label_name_to_id = {k.lower(): v for k, v in model.config.label2id.items()}
        if sorted(label_name_to_id.keys()) == sorted(label_list):
            label_to_id = {i: int(label_name_to_id[label_list[i]]) for i in range(num_labels)}
        else:
            logger.warning(
                "Your model seems to have been trained with labels, but they don't match the dataset: "
                f"model labels: {sorted(label_name_to_id.keys())}, dataset labels: {sorted(label_list)}."
                "\nIgnoring the model labels as a result.",
            )

    if label_to_id is not None:
        model.config.label2id = label_to_id
        model.config.id2label = {id: label for label, id in model.config.label2id.items()}
    elif not is_regression:
        model.config.label2id = {l: i for i, l in enumerate(label_list)}
        model.config.id2label = {id: label for label, id in model.config.label2id.items()}

    if configs.max_seq_length > tokenizer.model_max_length:
        logger.warning(
            f"The max_seq_length passed ({configs.max_seq_length}) is larger than the maximum length for the "
            f"model ({tokenizer.model_max_length}). Using max_seq_length={tokenizer.model_max_length}."
        )
    max_seq_length = min(configs.max_seq_length, tokenizer.model_max_length)

    def preprocess_function(examples):
        # Tokenize the texts
        args = (
            (examples[sentence1_key],) if sentence2_key is None else (examples[sentence1_key], examples[sentence2_key])
        )
        result = tokenizer(*args, padding=padding, max_length=max_seq_length, truncation=True)

        # Map labels to IDs (not necessary for GLUE tasks)
        if label_to_id is not None and "label" in examples:
            result["label"] = [(label_to_id[l] if l != -1 else -1) for l in examples["label"]]
        return result

    raw_datasets = raw_datasets.map(
        preprocess_function,
        batched=True,
        load_from_cache_file=True,
        desc="Running tokenizer on dataset",
    )

    def print_class_distribution(dataset, split_name):
        label_counts = Counter(dataset["label"])
        total = sum(label_counts.values())
        logger.info(f"Class distribution in {split_name} set:")
        for label, count in label_counts.items():
            logger.info(f"  Label {label}: {count} ({count / total:.2%})")

    if "validation" not in raw_datasets and "validation_matched" not in raw_datasets:
        raise ValueError("--do_eval requires a validation dataset")
    eval_dataset = raw_datasets["validation_matched" if configs.task_name == "mnli" else "validation"]
    print_class_distribution(eval_dataset, "validation")

    # Get the metric function
    metric = evaluate.load("glue", configs.task_name, cache_dir=configs.cache_dir)

    # You can define your custom compute_metrics function. It takes an `EvalPrediction` object (a namedtuple with a
    # predictions and label_ids field) and has to return a dictionary string to float.
    def compute_metrics(p: EvalPrediction):
        preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
        labels = p.label_ids
        if not training_args.eval_do_concat_batches:
            preds = np.concatenate(preds, axis=0)
            labels = np.concatenate(p.label_ids, axis=0)
        preds = np.squeeze(preds) if is_regression else np.argmax(preds, axis=1)
        result = metric.compute(predictions=preds, references=labels)
        if len(result) > 1:
            result["combined_score"] = np.mean(list(result.values())).item()
        return result

    # Data collator will default to DataCollatorWithPadding when the tokenizer is passed to Trainer, so we change it if
    # we already did the padding.
    data_collator = default_data_collator

    # Initialize our Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=None,
        eval_dataset=eval_dataset if training_args.do_eval else None,
        compute_metrics=compute_metrics,
        processing_class=tokenizer,
        data_collator=data_collator,
    )

    # Evaluation
    logger.info("*** Evaluate ***")

    # Loop to handle MNLI double evaluation (matched, mis-matched)
    tasks = [configs.task_name]
    eval_datasets = [eval_dataset]
    if configs.task_name == "mnli":
        tasks.append("mnli-mm")
        valid_mm_dataset = raw_datasets["validation_mismatched"]
        eval_datasets.append(valid_mm_dataset)
        combined = {}

    for eval_dataset, task in zip(eval_datasets, tasks):
        metrics = trainer.evaluate(eval_dataset=eval_dataset)

        max_eval_samples = len(eval_dataset)
        metrics["eval_samples"] = len(eval_dataset)

        if task == "mnli-mm":
            metrics = {k + "_mm": v for k, v in metrics.items()}
        if task is not None and "mnli" in task:
            combined.update(metrics)

        trainer.log_metrics("eval", metrics)

# Load pretrained model and tokenizer
tokenizer = BertTokenizer.from_pretrained(configs.output_dir)

evaluate_model(model, tokenizer)
evaluate_model(quantized_model, tokenizer)

***** eval metrics *****
  eval_accuracy               =     0.8603
  eval_combined_score         =     0.8811
  eval_f1                     =     0.9019
  eval_loss                   =     0.5857
  eval_model_preparation_time =     0.0019
  eval_runtime                = 0:00:54.66
  eval_samples                =        408
  eval_samples_per_second     =      7.464
  eval_steps_per_second       =      0.933


***** eval metrics *****
  eval_accuracy               =     0.8529
  eval_combined_score         =     0.8742
  eval_f1                     =     0.8955
  eval_loss                   =     0.4196
  eval_model_preparation_time =     0.0019
  eval_runtime                = 0:00:32.17
  eval_samples                =        408
  eval_samples_per_second     =     12.679
  eval_steps_per_second       =      1.585


## 2. Quantization and Inference with ORT ##
In this section, we will demonstrate how to export the PyTorch model to ONNX, quantize the exported ONNX model, and infererence the quantized model with ONNXRuntime.

### 2.1 Export to ONNX model and optimize with ONNXRuntime-tools
This step will export the PyTorch model to ONNX and then optimize the ONNX model with onnxruntime.

In [13]:
import onnxruntime
from itertools import chain
from onnxruntime.quantization import quant_pre_process
from fusion_options import FusionOptions

def export_onnx(model, tokenizer, output_onnx_path):
    from transformers.onnx.features import FeaturesManager
    onnx_config = FeaturesManager._SUPPORTED_MODEL_TYPE['bert']['sequence-classification'](training_args)
    dummy_inputs = onnx_config.generate_dummy_inputs(tokenizer, framework='pt')
    torch.onnx.export(model,
        (dummy_inputs,),
        f=output_onnx_path,
        input_names=list(onnx_config.inputs.keys()),
        output_names=list(onnx_config.outputs.keys()),
        dynamic_axes={name: axes for name, axes in chain(onnx_config.inputs.items(), onnx_config.outputs.items())},
        opset_version=17, 
        dynamo=False
    )


def preprocess_onnx(onnx_path, pre_onnx_path):
    # disable embedding layer norm optimization for better model size reduction
    opt_options = FusionOptions('bert')
    opt_options.enable_embed_layer_norm = False
    opt_options.intra_op_num_threads = 1
    opt_options.inter_op_num_threads = 1 

    quant_pre_process(onnx_path, 
                      pre_onnx_path, 
                      auto_merge=True,
                      optimization_options=opt_options)
    
# get model
tokenizer = BertTokenizer.from_pretrained(configs.output_dir)

model = BertForSequenceClassification.from_pretrained(configs.output_dir)
model.to(configs.device)

# convert to onnx
onnx_path = "bert_mrpc.onnx"
export_onnx(model, tokenizer, onnx_path)

# optimize model
preprocessed_model_path = "bert_mrpc_optimized.onnx"
preprocess_onnx(onnx_path, preprocessed_model_path)

C:\Users\18220\AppData\Local\Temp\ipykernel_14996\2888823050.py:10: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter will be the default. To switch now, set dynamo=True in torch.onnx.export. This new exporter supports features like exporting LLMs with DynamicCache. We encourage you to try it and share feedback to help improve the experience. Learn more about the new export logic: https://pytorch.org/docs/stable/onnx_dynamo.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html.
  torch.onnx.export(model,
C:\Users\18220\work\onnxruntime-inference-examples\jupyter_env\Lib\site-packages\transformers\modeling_attn_mask_utils.py:196: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the

### 2.2 Quantize ONNX model
We will call [onnxruntime.quantization.quantize_dynamic](https://github.com/microsoft/onnxruntime/blob/main/onnxruntime/python/tools/quantization/README.md) to apply quantization on the HuggingFace BERT model. It supports dynamic quantization with IntegerOps and static quantization with QLinearOps. For activation ONNXRuntime supports only uint8 format for now, and for weight ONNXRuntime supports both int8 and uint8 format.

We apply dynamic quantization for BERT model and use int8 for weight.

In [14]:
from onnxruntime.quantization import quantize_dynamic, QuantType
import onnx

def quantize_onnx(onnx_path, quant_onnx_path):
    quantize_dynamic(onnx_path, 
                     quant_onnx_path, 
                     weight_type=QuantType.QInt8,
                     extra_options={'DefaultTensorType': onnx.TensorProto.FLOAT})
    
# quantize model
quantized_model_path = "bert_mrpc_quant.onnx"
quantize_onnx(preprocessed_model_path, quantized_model_path)

print('ONNX full precision model size (MB):', os.path.getsize(onnx_path)/(1024*1024))

print('ONNX quantized model size (MB):', os.path.getsize(quantized_model_path)/(1024*1024))
    

ONNX full precision model size (MB): 417.848087310791
ONNX quantized model size (MB): 105.07765865325928


### 2.3 Evaluate ONNX quantization performance and accuracy

In this step, we will evalute OnnxRuntime quantization with GLUE data set.

In [15]:
import time
from torch.utils.data import DataLoader


def evaluate_onnx(onnx_path, tokenizer):
    # load dataset
    raw_datasets = load_dataset("glue", configs.task_name, cache_dir=configs.cache_dir)
    sentence1_key, sentence2_key = task_to_keys[configs.task_name]

    def preprocess_function(examples):
        # Tokenize the texts
        args = ((examples[sentence1_key],) if sentence2_key is None else (examples[sentence1_key], examples[sentence2_key]))
        result = tokenizer(*args, padding='max_length', max_length=configs.max_seq_length, truncation=True)

        return result

    eval_dataset = raw_datasets["validation"].map(
        preprocess_function,
        batched=True,
        load_from_cache_file=True,
        desc="Running tokenizer on validation dataset",
    )

    # create onnx runtime session
    sess_options = onnxruntime.SessionOptions()
    sess_options.graph_optimization_level = onnxruntime.GraphOptimizationLevel.ORT_ENABLE_ALL
    ort_session = onnxruntime.InferenceSession(onnx_path, sess_options,  providers=['CPUExecutionProvider'])

    # evaluation
    metric = evaluate.load("glue", configs.task_name, cache_dir=configs.cache_dir)

    def onnx_eval_step(batch):
        ort_inputs = {k: v.cpu().numpy() for k, v in batch.items() if k in ['input_ids', 'attention_mask', 'token_type_ids']}
        ort_outs = ort_session.run(None, ort_inputs)
        return torch.tensor(ort_outs[0])

    dataloader = DataLoader(
        eval_dataset,
        batch_size=training_args.per_device_eval_batch_size,
        collate_fn=default_data_collator,
        shuffle=True
    )

    for batch in dataloader:
        logits = onnx_eval_step(batch)
        labels = batch['labels']
        metric.add_batch(predictions=logits.argmax(dim=-1), references=labels)

    eval_metric = metric.compute()
    print(f"ONNX model evaluation results: {eval_metric}")


def time_ort_model_evaluation(model_path, tokenizer):
    eval_start_time = time.time()
    evaluate_onnx(model_path, tokenizer)
    eval_end_time = time.time()
    eval_duration_time = eval_end_time - eval_start_time
    print("Evaluate total time (seconds): {0:.1f}".format(eval_duration_time))

time_ort_model_evaluation(onnx_path, tokenizer)
time_ort_model_evaluation(quantized_model_path, tokenizer)

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Running tokenizer on validation dataset:   0%|          | 0/408 [00:00<?, ? examples/s]

ONNX model evaluation results: {'accuracy': 0.8602941176470589, 'f1': 0.9018932874354562}
Evaluate total time (seconds): 63.1
ONNX model evaluation results: {'accuracy': 0.8553921568627451, 'f1': 0.9005059021922428}
Evaluate total time (seconds): 38.3


## 3 Summary
In this tutorial, we demonstrated how to quantize a fine-tuned BERT model for MPRC task on GLUE data set. Let's summarize the main metrics of quantization.

### Model Size
PyTorch quantizes torch.nn.Linear modules only and reduce the model from 438 MB to 173 MB. OnnxRuntime quantizes not only Linear(MatMul), but also the embedding layer. It achieves almost the ideal model size reduction with quantization.

| Engine | Full Precision(MB) | Quantized(MB) |
| --- | --- | --- |
| PyTorch 2.9 | 417.7 | 173.1 |
| ORT 1.23 | 417.8 | 105.1 |

### Accuracy
OnnxRuntime achieves a little bit better accuracy and F1 score, even though it has small model size.

| Metrics | Full Precision | PyTorch 2.9 Quantization | ORT 1.23 Quantization |
| --- | --- | --- | --- |
| Accuracy | 0.8603 | 0.8529 | 0.8554 |
| F1 score | 0.9019 | 0.8955 | 0.9005 |
| Acc and F1 | 0.8811 | 0.8742 | 0.8780 |

### Performance

The evaluation data set has 408 sample. Table below shows the performance on **Azure VM: Standard E4ds_v4 (4 vcpus, 32 GiB memory)**. Comparing with PyTorch full precision, PyTorch quantization achieves ~1.70x speedup, and ORT quantization achieves ~1.65x speedup. 
You can run the [benchmark.py](https://github.com/microsoft/onnxruntime/blob/master/onnxruntime/python/tools/transformers/benchmark.py) for comparison on more models.

|Engine | Full Precision Latency(s) | Quantized(s) |
| --- | --- | --- |
| PyTorch 2.9 | 54.7 | 32.2 |
| ORT 1.23 | 63.1 | 38.3 |